
# TRNG Project



## background
We read some articles and start with several main ideas, here a brief explanation-
1. The first one is to use the diffrent oscilator jitters for entropy source. we took one fast oscilator (32Mhz) for a COUNTER peripheral that inside the XMEGA mcu and we talk anouther independent oscillator that runs for RTC (real time counter). We use the jitter and the noise f this oscillators as an entropy source, After each OVF of the RTC we took the COUNTER value (the 8 lsb bits).

2. we took the ADC peripheral of the mcu and want to use some analog entropy (sensors) and digitized them to get random bits. we try two sources that we have inside the MCU - VCC/10, TEMP SENSOR. we see that in room conditions the temp gives us ~ 5 bits if entopy for sample, and vcc/10 ~ 3 bits per sample, we take it as an start point and trying to achive the best results. 

## 1) Build firmware

In [1]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange

scope = cw.scope()
target = cw.target(scope, cw.targets.SimpleSerial2)
scope.default_setup()

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 45898501                  to 59457014                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 3904277                   to 29538459                 
scope.clock.adc_rate                     changed from 3904277.0                 to 29538459.0               
scope.clock.clkgen_

In [ ]:
%%bash
make -f ../makefile PLATFORM=CWLITEXMEGA  SS_VER=SS_VER_2_1

No CRYPTO_TARGET passed - defaulting to TINYAES128C
Building for platform CWLITEXMEGA with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
Blank crypto options, building for AES128
.
Welcome to another exciting ChipWhisperer target build!!
avr-gcc (GCC) 7.3.0
Copyright (C) 2017 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEXMEGA 
.
Compiling:
-en     trng.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/hal/hal.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/hal//xmega/XMEGA_AES_driver.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/hal//xmega/uart.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/hal//xmega/usart_driver.c ...
-e Done!
.
Compiling:
-en     ../../../firmware/mcu/hal//

## 2) Program target (XMEGA on CW303)

In [4]:
hex = "output-CWLITEXMEGA.hex"
cw.program_target(scope, cw.programmers.XMEGAProgrammer, hex)

XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 4415 bytes


## 3) Acquire random bits - Approach A: RTC vs TCC jitter

## senity check

In [ ]:
import numpy as np
import time
import logging   # <-- add this

# Silence ChipWhisperer "Target" warnings
logging.getLogger("ChipWhisperer Target").setLevel(logging.ERROR)

CHUNK = 100            # safe per-request size
I = 2
N = I*CHUNK               # total bytes you want
buf = bytearray()
scmd = 2
# Loop until we have N bytes
remaining = N
while remaining > 0:
    target.flush()
    want = min(CHUNK, remaining)


    # Ask target for that chunk
    target.simpleserial_write('b',bytes([scmd, want]))

    # Read exactly 'want' bytes (no ack check until final)
    chunk = target.simpleserial_read('r', want, timeout=20000)
    if chunk is None or len(chunk) != want:
        print("Bad block, skipping…")
        continue
    # if chunk is None:
    #     raise RuntimeError("Timeout waiting for target")

    buf.extend(chunk)
    remaining -= want
    # time.sleep(1)   # 10 ms gap between chunks


# ---------- Analysis ----------
data = np.frombuffer(buf, dtype=np.uint8)
biases = []

for b in range(8):
    ones = np.count_nonzero((data >> b) & 1)
    p1 = ones / len(data)
    biases.append(p1)
    print(f"bit {b}: p1={p1:.5f}")

# ---------- Plot ----------
plt.figure(figsize=(7,4))
plt.bar(range(8), biases, color="royalblue")
plt.axhline(0.5, color="red", linestyle="--", label="ideal = 0.5")
plt.xticks(range(8), [f"bit {i}" for i in range(8)])
plt.ylabel("Probability of 1")
plt.title("Bit bias per position (from 10,000 bytes)")
plt.legend()
plt.show()


# ===== 3. Byte value distribution =====
values, counts = np.unique(data, return_counts=True)
probs = counts / len(data)

print("\nByte distribution stats:")
print(f"Min prob: {probs.min():.4f}, Max prob: {probs.max():.4f}, Expected = {1/256:.4f}")

plt.hist(data, bins=256, range=(0, 255), density=True, color='skyblue')
plt.axhline(1/256, color='r', linestyle='--', label="Uniform expected")
plt.xlabel("Byte value (0–255)")
plt.ylabel("Probability")
plt.title("Distribution of TRNG Byte Values")
plt.legend()
plt.show()

KeyboardInterrupt: 

## 6) Cleanup / disconnect (optional)

In [245]:

try:
    target.dis()
except Exception:
    pass
try:
    scope.dis()
except Exception:
    pass
print("Disconnected.")


Disconnected.
